# Atividade · Regressão logística na prática
**Fundamentos de Machine Learning · Aula 3 · Tempo sugerido: 1 hora**

Agora é sua vez. Aplique a receita do notebook de passo a passo a uma base nova.

**Como resolver**
- Complete cada `___` marcado com `# COMPLETE`.
- Execute as células de **verificação**. Se aparecer ✔, siga em frente.
- Responda as **questões abertas** na célula de texto logo abaixo de cada uma.

| Parte | Conteúdo | Tempo |
|---|---|---|
| 1 | Carregar o CSV | 5 min |
| 2 | Explorar e limpar a base | 10 min |
| 3 | Por que não a regressão linear | 5 min |
| 4 | Treinar com uma feature | 10 min |
| 5 | Probabilidades, limiar e erros | 10 min |
| 6 | Log-loss | 10 min |
| 7 | Duas features | 10 min |

## A base de dados: *Breast Cancer Wisconsin (Diagnostic)*

A base vem do **UCI Machine Learning Repository** (dataset id 17, licença CC BY 4.0) e também está no Kaggle (*uciml/breast-cancer-wisconsin-data*). Cada linha é um **tumor de mama** examinado por punção. As 30 features descrevem os núcleos das células vistos na imagem do exame.

| Coluna | Significado |
|---|---|
| `id` | identificador do exame (não é feature) |
| `diagnosis` | **alvo**: `M` = maligno, `B` = benigno |
| `radius_mean` | raio médio dos núcleos |
| `texture_mean` | textura média (variação dos tons de cinza) |
| `concave points_mean` | número médio de pontos côncavos no contorno dos núcleos |
| ... | outras 27 medidas: cada característica aparece como média (`_mean`), erro padrão (`_se`) e pior valor (`_worst`) |

**Pergunta:** a partir das medidas da imagem, qual a probabilidade de o tumor ser maligno?

**Como obter o arquivo:** baixe `data.csv` do Kaggle e salve-o em `data/` com o nome `breast-cancer.csv`. Se preferir a fonte oficial da UCI, rode a célula opcional abaixo: ela usa o pacote `ucimlrepo` e grava um CSV no mesmo formato.

In [ ]:
# # OPCIONAL: baixar da UCI e salvar no formato do Kaggle (requer: pip install ucimlrepo)
# from ucimlrepo import fetch_ucirepo
# bc = fetch_ucirepo(id=17)
# tab = bc.data.original
# nomes = {}
# for c in tab.columns:                      # radius1 -> radius_mean, radius2 -> radius_se, radius3 -> radius_worst
#     base, sufixo = c[:-1], c[-1]
#     if sufixo in '123' and c not in ('ID', 'Diagnosis'):
#         nomes[c] = base.replace('_', ' ') + {'1': '_mean', '2': '_se', '3': '_worst'}[sufixo]
# tab = tab.rename(columns={**nomes, 'ID': 'id', 'Diagnosis': 'diagnosis'})
# tab.to_csv('breast-cancer.csv', index=False)

## Parte 0 · Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import log_loss

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

## Parte 1 · Carregar o CSV

**Tarefa 1.** Leia o arquivo e mostre as 5 primeiras linhas.

In [ ]:
df = pd.read_csv(___)          # COMPLETE: caminho do arquivo
df.___()                       # COMPLETE: 5 primeiras linhas

In [ ]:
# Verificação
assert 'diagnosis' in df.columns and 'radius_mean' in df.columns and len(df) == 569
print('✔ Tarefa 1 ok')

## Parte 2 · Explorar e limpar a base

**Tarefa 2.** Mostre o tamanho da base e quantos tumores há em cada classe de `diagnosis`.

In [ ]:
print(df.___)                       # COMPLETE: (linhas, colunas)
df['diagnosis'].___()               # COMPLETE: contagem por classe

**Tarefa 3.** Limpe a base e guarde o resultado em `df_limpo`:
1. remova a coluna `id` e qualquer coluna cujo nome comece com `Unnamed` (o CSV do Kaggle termina com uma coluna vazia, `Unnamed: 32`);
2. crie a coluna `alvo` com **1 para maligno (`M`)** e **0 para benigno (`B`)**, usando `.map`.

In [ ]:
remover = ['id'] + [c for c in df.columns if c.startswith(___)]   # COMPLETE
df_limpo = df.drop(columns=remover)
df_limpo['alvo'] = df_limpo['diagnosis'].map(___)                  # COMPLETE: dicionário M/B -> 1/0
df_limpo['alvo'].value_counts()

In [ ]:
# Verificação
assert 'id' not in df_limpo and not any(c.startswith('Unnamed') for c in df_limpo.columns)
assert df_limpo['alvo'].sum() == 212 and df_limpo.isna().sum().sum() == 0
print('✔ Tarefa 3 ok')

**Tarefa 4.** Compare a média de `radius_mean`, `texture_mean` e `concave points_mean` entre as duas classes e faça o histograma de `radius_mean` separado por classe.

In [ ]:
cols = ['radius_mean', 'texture_mean', 'concave points_mean']
display(df_limpo.groupby(___)[cols].mean().round(3))             # COMPLETE: agrupar pelo alvo

plt.figure(figsize=(8, 4))
plt.hist(df_limpo.loc[df_limpo['alvo'] == 0, 'radius_mean'], bins=30, alpha=0.6, label='benigno (0)')
plt.hist(df_limpo.loc[df_limpo['alvo'] == ___, 'radius_mean'], bins=30, alpha=0.6, label='maligno (1)')   # COMPLETE
plt.xlabel('radius_mean'); plt.legend()
plt.show()

> **Questão aberta 1.** Qual das três features parece separar melhor tumores benignos e malignos? Os dois histogramas de `radius_mean` se sobrepõem? O que essa sobreposição significa para um classificador que usa só o raio?

*Sua resposta:*

## Parte 3 · Por que não a regressão linear?

**Tarefa 5.** Ajuste uma `LinearRegression` de `alvo` em função de `radius_mean` (com a base inteira) e conte quantas previsões ficam abaixo de 0 e acima de 1.

In [ ]:
reta = LinearRegression().fit(df_limpo[[___]], df_limpo[___])   # COMPLETE
prev = reta.predict(df_limpo[['radius_mean']])

print(f'menor: {prev.min():.2f} | maior: {prev.max():.2f}')
print('abaixo de 0:', (prev < 0).sum(), '| acima de 1:', ___)     # COMPLETE

> **Questão aberta 2.** O que significaria, para um médico, uma "probabilidade de malignidade" de −0,3 ou de 1,7? Que propriedade a função sigmoide garante e a reta não?

*Sua resposta:*

## Parte 4 · Treinar com uma feature

**Tarefa 6.** Monte `X` só com `radius_mean` e `y` com `alvo`. Separe 20% para teste, com `random_state=42` e **estratificando pelo alvo**.

In [ ]:
X = df_limpo[___]                      # COMPLETE: DataFrame com uma coluna (2D)
y = df_limpo['alvo']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=___)     # COMPLETE
print('Treino:', X_train.shape, '| Teste:', X_test.shape)

**Tarefa 7.** Treine a `LogisticRegression`, guarde `a` e `b` e calcule o centro da curva `x* = −a/b`.

In [ ]:
modelo = ___                           # COMPLETE: crie o modelo
modelo.fit(___, ___)                   # COMPLETE: treine com o conjunto de treino

a = modelo.___[0]                      # COMPLETE: intercepto
b = modelo.___[0][0]                   # COMPLETE: coeficiente
x_estrela = ___                        # COMPLETE
print(f'a = {a:.3f} | b = {b:.3f} | x* = {x_estrela:.2f}')

In [ ]:
# Verificação
assert X.ndim == 2 and len(X_test) == 114 and y_test.sum() == 42, 'confira a separação (test_size, random_state, stratify)'
assert b > 0 and 13 < x_estrela < 17
print('✔ Tarefas 6 e 7 ok')

**Tarefa 8.** Desenhe a sigmoide ajustada sobre os pontos de treino, com a linha de 0,5 e a linha vertical em `x*`.

In [ ]:
x_grade = np.linspace(5, 30, 200)

plt.figure(figsize=(10, 5))
plt.scatter(X_train['radius_mean'], y_train, alpha=0.3, s=15, label='treino')
plt.plot(x_grade, ___, color='red', linewidth=2.5, label='regressão logística')   # COMPLETE: σ(a + b·x)
plt.axhline(0.5, color='gray', linestyle='--')
plt.axvline(___, color='black', linestyle=':', label='x*')                         # COMPLETE
plt.xlabel('radius_mean'); plt.ylabel('P(maligno)'); plt.legend()
plt.show()

> **Questão aberta 3.** Interprete `x*` em uma frase. O que aconteceria com a curva se `b` fosse duas vezes maior (com `x*` igual)?

*Sua resposta:*

## Parte 5 · Probabilidades, limiar e erros

**Tarefa 9.** Calcule a probabilidade de malignidade para tumores com raio 12, 15 e 18 usando `predict_proba`, e confira o raio 15 na mão com a função `sigmoide`.

In [ ]:
novos = pd.DataFrame({'radius_mean': [12, 15, 18]})
novos['P(maligno)'] = modelo.___(novos)[:, ___].round(3)      # COMPLETE
display(novos)

p15_mao = ___                                                 # COMPLETE: σ(a + b·15)
print(f'raio 15 na mão: {p15_mao:.3f}')

In [ ]:
# Verificação
assert abs(p15_mao - novos.loc[1, 'P(maligno)']) < 1e-3
print('✔ Tarefa 9 ok')

**Tarefa 10.** Complete a função `contar_erros`, que recebe um limiar e devolve, no conjunto de **teste**:
- `maligno_nao_detectado`: tumores malignos (y = 1) classificados como benignos (previsto 0);
- `alarme_falso`: tumores benignos (y = 0) classificados como malignos (previsto 1).

Depois rode para os limiares 0,5, 0,3 e 0,2.

In [ ]:
p_test = modelo.predict_proba(X_test)[:, 1]

def contar_erros(limiar):
    previsto = (p_test >= ___).astype(int)                          # COMPLETE
    return {
        'limiar': limiar,
        'acertos': int((previsto == y_test).sum()),
        'maligno_nao_detectado': int(((previsto == 0) & (y_test == 1)).sum()),
        'alarme_falso': int(___),                                    # COMPLETE
    }

pd.DataFrame([contar_erros(l) for l in [0.5, 0.3, 0.2]])

In [ ]:
# Verificação
r = contar_erros(0.5)
assert r['maligno_nao_detectado'] + r['alarme_falso'] + r['acertos'] == len(y_test)
assert contar_erros(0.2)['maligno_nao_detectado'] <= r['maligno_nao_detectado']
print('✔ Tarefa 10 ok')

> **Questão aberta 4.** Nesta aplicação, qual dos dois erros é mais grave? Com base na tabela, que limiar você escolheria e o que você abre mão ao escolhê-lo?

*Sua resposta:*

## Parte 6 · Log-loss

**Tarefa 11.** Calcule a log-loss do conjunto de teste **na mão**, com a fórmula da aula, e confira com `log_loss` do scikit-learn. Calcule também a log-loss de um modelo ingênuo que dá a todos os tumores a probabilidade `y_train.mean()`.

In [ ]:
J_mao = ___                                                   # COMPLETE: −média[y·log p + (1−y)·log(1−p)]
J_sk = log_loss(y_test, p_test)

p_ingenuo = np.full(len(y_test), ___)                         # COMPLETE
J_ingenuo = log_loss(y_test, p_ingenuo)
print(f'na mão: {J_mao:.4f} | sklearn: {J_sk:.4f} | ingênuo: {J_ingenuo:.4f}')

In [ ]:
# Verificação
assert abs(J_mao - J_sk) < 1e-6 and J_sk < J_ingenuo
print('✔ Tarefa 11 ok')

## Parte 7 · Duas features

**Tarefa 12.** Treine `modelo2` com `radius_mean` e `texture_mean` (mesma separação: `test_size=0.2`, `random_state=42`, `stratify=y`). Depois desenhe a fronteira de decisão no plano, isolando `texture_mean` na equação `a + b₁·radius + b₂·texture = 0`.

In [ ]:
X2 = df_limpo[['radius_mean', 'texture_mean']]
X2_train, X2_test, y_train, y_test = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)

modelo2 = LogisticRegression().fit(___, ___)                  # COMPLETE
a2 = modelo2.intercept_[0]
b1, b2 = modelo2.coef_[0]
print(f'a = {a2:.3f} | b1 (raio) = {b1:.3f} | b2 (textura) = {b2:.3f}')

raio = np.linspace(6, 28, 100)
textura_fronteira = ___                                       # COMPLETE: isole a textura

plt.figure(figsize=(9, 6))
plt.scatter(X2_test['radius_mean'], X2_test['texture_mean'], c=y_test, cmap='coolwarm', edgecolor='k', alpha=0.8)
plt.plot(raio, textura_fronteira, 'k--', label='fronteira (p = 0,5)')
plt.ylim(8, 40)
plt.xlabel('radius_mean'); plt.ylabel('texture_mean'); plt.legend()
plt.show()

In [ ]:
# Verificação
p2 = modelo2.predict_proba(X2_test)[:, 1]
print(f'log-loss teste: 1 feature = {J_sk:.4f} | 2 features = {log_loss(y_test, p2):.4f}')
print(f'acertos (0,5) : 1 feature = {(modelo.predict(X_test) == y_test).sum()} | 2 features = {(modelo2.predict(X2_test) == y_test).sum()}')
assert b1 > 0 and b2 > 0
print('✔ Tarefa 12 ok')

> **Questão aberta 5.** A fronteira é uma reta com inclinação negativa. Explique, em termos de raio e textura, o que essa inclinação significa. A segunda feature ajudou? Use a comparação impressa na verificação.

*Sua resposta:*